### LLM As Judge

---
If Ollama natively supports the OpenAI /v1 format (which we are using to bypass the proxy), it begs the question: Why include LiteLLM in the stack at all?

In a purely local, single-user script (like the Ragas evaluation you are running), LiteLLM is an unnecessary middleman. That is exactly why we bypassed it in the lab scripts to save your computer's RAM and avoid network routing errors.

However, in a Production Agentic Stack, LiteLLM serves a crucial role as an "LLM Gateway." Here is exactly what LiteLLM provides and why it is typically included in enterprise architectures:

1. Code-Free Observability (The Langfuse Connection)
If your Python scripts or n8n workflows talk directly to Ollama, Ollama simply generates text and forgets about it. You have no record of what was said, how long it took, or how many tokens were used.

LiteLLM acts as a tollbooth. When you put LiteLLM in the middle, it automatically captures every request, calculates the token metrics, and pushes that data to Langfuse (or other tracing tools). This gives you a dashboard to monitor latency, track token usage, and calculate the "cost" of your local models as if they were cloud models—all without writing complex tracking code in your main application.

2. Universal API Routing (The "Drop-in" Architecture)
In your curriculum, you are building workflows meant to simulate enterprise environments. An enterprise might use a local model (Ollama) for reading internal HR documents, but use a cloud model (OpenAI's GPT-4o) for writing client-facing emails.

LiteLLM allows your application (whether it is n8n, CrewAI, or AutoGen) to only ever point to one single URL: http://localhost:4000.

You just tell LiteLLM: "Hey, use the ollama/llama3 model." -> LiteLLM translates and routes it locally.

Later, you tell LiteLLM: "Hey, use the gpt-4 model." -> LiteLLM securely attaches your API key and routes it to the cloud.
Your main application code never has to change, update its SDKs, or manage API keys.

3. Fallbacks and Load Balancing
In a production environment, local models can crash, or cloud APIs can hit rate limits. LiteLLM is designed to handle this seamlessly. You can configure LiteLLM so that if Ollama goes offline or takes too long to respond, the request automatically "falls back" to Groq or OpenAI. The end-user (or the AI Agent) never even notices the failure.

Summary: Why We Bypassed It for Ragas
We bypassed LiteLLM for the Ragas Evaluation Lab because:

Ragas "Judge" LLMs are extremely heavy: They generate massive JSON objects and send huge context windows.

Local Hardware Constraints: Routing massive payloads through an extra Docker container (LiteLLM) while simultaneously running the LLM (Ollama) on a single laptop causes memory bottlenecks and timeouts.

The Rule of Thumb: Use direct-to-Ollama when you are doing heavy backend data processing or local evaluation where speed and RAM are the only priorities. Use LiteLLM when you have a user-facing application (like an n8n chatbot) where you need to track token usage, monitor user behavior, or easily swap out models behind the scenes


#### Assignment
Understand the metrics from: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/

In [2]:
import os
from calendar import error

# Install it first if you don't have it
#!pip install nest_asyncio

# Apply the patch to allow nested async loops
import nest_asyncio
nest_asyncio.apply()

# SCORE METRIC LIBRARIES
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
    ResponseGroundedness,
    FactualCorrectness,
)

from datasets import Dataset
from pandas import DataFrame
from ragas import evaluate
from ragas.run_config import RunConfig

print(os.environ.get("OPENAI_MODEL"))
print(os.environ.get("MODEL"))
print(os.environ.get("LLM_MODEL"))

None
None
None


In [9]:
dataset = Dataset.from_dict({
    "question": [
        "What is Kubernetes?",
        "Who created Docker?"
    ],

    "answer": [
        "Kubernetes is a container orchestration platform.",
        "Docker was created by Solomon Hykes."
    ],

    "contexts": [
        [
            "Kubernetes is an open-source container orchestration platform."
        ],
        [
            "Docker was created by Solomon Hykes."
        ]
    ]
})

display(DataFrame(dataset))

,question,answer,contexts
0,What is Kubernetes?,Kubernetes is a container orchestration platform.,[Kubernetes is an open-source container orches...
1,Who created Docker?,Docker was created by Solomon Hykes.,[Docker was created by Solomon Hykes.]


#### Initialize Various Models

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings


'''
    * Used only for proxying via LiteLLM
    * Primarily in production to ensure:
    ** Tracing,
    ** Logging,
    ** Load-balancing across multiple LLMs
'''
local_judge_llm_PROXIED = ChatOpenAI(
    model="ollama/llama3",
    api_key="anything",
    base_url="http://localhost:4000",
    timeout=300,
)


# Initialize your local Embedding Model
# local_embeddings = OpenAIEmbeddings(
#     model="nomic-embed-text",
#     base_url="http://localhost:11434/v1",
#     api_key="anything",
# )

# Ensure the base_url includes /v1
local_embeddings_ollama_nomic = OllamaEmbeddings(
    model="nomic-embed-text",  # Use your specific model name here
    base_url="http://localhost:11434",
)

# Update your existing judge_model with these arguments:
local_judge_llm_llama3 = ChatOpenAI(
    model="llama3",
    base_url="http://localhost:11434/v1",
    api_key="anything",
    # Crucial: Must be 0
    temperature=0,
    # Forces JSON API mode
    model_kwargs={"response_format": {"type": "json_object"}}
)

#### TEST LOCAL SETUP
* OLLAMA
    * **```nomic-embed-text```** as Embedding Model (```OllamaEmbeddings```)
    * **```llama3```** as the LLM for Generation (wrapped underd ```ChatOpenAI```)

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import OllamaEmbeddings

# Ensure the base_url includes /v1
# local_embeddings_ollama_nomic = OllamaEmbeddings(
#     model="nomic-embed-text",  # Use your specific model name here
#     base_url="http://localhost:11434",
# )

# Now test it again
vector = local_embeddings_ollama_nomic.embed_query("This is a test.")
display("Vector Dimensions: " + str(len(vector)))

'Vector Dimensions: 768'

In [7]:
'''
!! @SKIP !!
Should take ~15 secs (locally)
'''
local_judge_llm_llama3.invoke("What is Kubernetes?")

AIMessage(content='{}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 14, 'total_tokens': 16, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'llama3', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-596', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--019ee990-a349-76c2-b035-ed8d341a4fa0-0', usage_metadata={'input_tokens': 14, 'output_tokens': 2, 'total_tokens': 16, 'input_token_details': {}, 'output_token_details': {}})

In [8]:
print(local_judge_llm_llama3.model_name)
print(local_embeddings_ollama_nomic.model)

llama3
nomic-embed-text


#### Evaluation Basic
* Metrics
    * **Faithfulness** - adherence to the context

In [8]:
# 1. Create a custom configuration for local models
local_run_config = RunConfig(
    timeout=3000,   # Give Ollama up to few minutes to reply per prompt
    max_workers=1   # Force Ragas to evaluate exactly 1 row at a time
)

results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
    ],
    llm=local_judge_llm_llama3,
    run_config=local_run_config
)
df = results.to_pandas()

print("Only One metric - Failthfulness")
display(df)

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,faithfulness
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is a container orchestration platform.,1.0
1,Who created Docker?,[Docker was created by Solomon Hykes.],Docker was created by Solomon Hykes.,1.0


#### EVALUATION - Multiple - 1
#### DataSet2
* Models
    * **```nomic-embed-text```** as Embedding Model (```OllamaEmbeddings```)
    * **```llama3```** as the LLM for Generation (wrapped underd ```ChatOpenAI```)
* Metrics
    * **Faithfulness** -
    * **Answer Relevancy** -
    * **Context Precision** -
    * **Context Recall** -

In [10]:
dataset2 = Dataset.from_dict({
    "question": [
        "What is Kubernetes?",
        "Who created Docker?"
    ],

    "answer": [
        "Docker was created by Solomon Hykes.",
        "Kubernetes is a container orchestration platform.",
    ],

    "contexts": [
        [
            "Kubernetes is an open-source container orchestration platform.",
            "Docker was created by Solomon Hykes."
        ],
        [
            "Kubernetes is an open-source container orchestration platform.",
            "Docker was created by Solomon Hykes."
        ]
    ],

    "reference": [
        "Kubernetes is an open-source container orchestration platform.",
        "Docker was created by Solomon Hykes."
    ]
})
display(DataFrame(dataset2))

,question,answer,contexts,reference
0,What is Kubernetes?,Docker was created by Solomon Hykes.,[Kubernetes is an open-source container orches...,Kubernetes is an open-source container orchest...
1,Who created Docker?,Kubernetes is a container orchestration platform.,[Kubernetes is an open-source container orches...,Docker was created by Solomon Hykes.


In [12]:
#  Create a custom configuration for local models
local_run_config = RunConfig(
    timeout=3000,   # Give Ollama up to few minutes to reply per prompt
    max_workers=1   # Force Ragas to evaluate exactly 1 row at a time
)

results2 = evaluate(
    dataset=dataset2,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision,
        answer_correctness
        #ResponseGroundedness,
        #FactualCorrectness
    ],
    llm=local_judge_llm_llama3,
    embeddings=local_embeddings_ollama_nomic,
    run_config=local_run_config
)
df = results2.to_pandas()
print("------- Multiple Metrics -------")
display(df)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

------- Multiple Metrics -------


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision,answer_correctness
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Docker was created by Solomon Hykes.,Kubernetes is an open-source container orchest...,0.5,0.647856,1.0,1.0,0.491517
1,Who created Docker?,[Kubernetes is an open-source container orches...,Kubernetes is a container orchestration platform.,Docker was created by Solomon Hykes.,0.5,0.623212,1.0,0.5,0.790099


#### SCORE EXPLANATION

The reason these Ragas metrics seem counter-intuitive—giving seemingly "high" scores to completely wrong answers—lies in how each metric is strictly isolated to measure just one specific part of the RAG pipeline.

Here is exactly why each metric is giving you these numbers based on your "swapped" dataset:

1. **Answer Relevancy** (~0.64)
You might expect this to be 0.0 since the answer has nothing to do with the question. However, Ragas calculates Answer Relevancy using Embedding Cosine Similarity.

How it works: Ragas takes the AI's response ("Docker was created by Solomon Hykes") and asks the Judge LLM to reverse-engineer a question for it (e.g., it generates: "Who created Docker?").

It then uses your embedding model (Nomic) to calculate the distance between the generated question ("Who created Docker?") and the actual user_input ("What is Kubernetes?").

Why it's 0.6: In the high-dimensional space of text embeddings, "Docker" and "Kubernetes" are highly related concepts (DevOps, containerization). Therefore, their vectors are somewhat close. A score of ~0.6 is actually a fairly low score in embedding space (which usually clusters highly related text between 0.8 and 1.0). It rarely drops to 0.0 unless the topics are entirely unrelated (like Kubernetes vs. baking a cake).

2. **Faithfulness** (0.5)
Faithfulness does not care about the user's question. It only checks for hallucinations based on the provided context.

How it works: It asks: "Can the claims made in the response be directly inferred from the retrieved_contexts?"

Why it's > 0: Your response is "Docker was created by Solomon Hykes." If you look at your retrieved_contexts array, that exact sentence is in there! The Judge LLM sees the claim in the answer, finds it in the context, and says, "This is not a hallucination, it is faithful to the source material."

3. **Context Recall** (1.0)
Context Recall completely ignores the AI's generated response. It strictly evaluates your Retriever.

How it works: It asks: "Does the retrieved_contexts contain all the information needed to match the reference (ground truth)?"

Why it's 1.0: Your reference for row 0 is "Kubernetes is an open-source..." and that exact string is present in your contexts list. Therefore, the retriever successfully "recalled" 100% of the necessary ground truth information.

4. **Context Precision** (1.0 and 0.5)
Context precision checks if the most relevant chunks in your context list are ranked at the top. Since you provided the exact same contexts array for both questions, but the questions and references differed, the precision shifts based on whether the "Docker" chunk or the "Kubernetes" chunk was evaluated as the primary target for that specific row.

#### The Takeaway
Your dataset perfectly illustrates why you cannot rely on a single metric in Generative AI.

The system was highly Faithful (it didn't hallucinate outside the given text).

The retriever had perfect Recall (it fetched the right documents).

But the overall system failed because the Answer Relevancy was heavily degraded.

To catch this failure in a real pipeline, you would look at the combination of these scores (or specifically, use an Answer Correctness metric, which strictly compares the generated answer directly against the ground truth reference—that would score a 0.0 here).

#### EVALUATION - Multiple - 1
#### DataSet 3
* Models
    * **```nomic-embed-text```** as Embedding Model (```OllamaEmbeddings```)
    * **```llama3```** as the LLM for Generation (wrapped underd ```ChatOpenAI```)
* Metrics
    * **Faithfulness** -
    * **Answer Relevancy** -
    * **Context Precision** -
    * **Context Recall** -

In [11]:
data = {
    'question': [
        'What is Kubernetes?',
        'What is Docker?'
    ],
    'answer': [
        'Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications.',
        # 'Docker is a platform designed to help developers build, share, and run modern applications.'
        'A Docker is a device that helps ship dock at the port.'
    ],
    'contexts': [
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.'],
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.']
    ],
    'reference': [
        'Kubernetes is an open-source system for managing containerized applications.',
        'Docker is a platform for building and running containers.'
    ]
}

dataset3 = Dataset.from_dict(data)
display(DataFrame(dataset3))

,question,answer,contexts,reference
0,What is Kubernetes?,Kubernetes is an open-source system for automa...,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for managi...
1,What is Docker?,A Docker is a device that helps ship dock at t...,[Kubernetes is an open-source container orches...,Docker is a platform for building and running ...


In [12]:
# Are you sure!
is_to_be_run = bool(input("Do you really want to re-run?"))

if not is_to_be_run:
    raise error()

#  Create a custom configuration for local models
local_run_config = RunConfig(
    timeout=3000,   # Give Ollama up to few minutes to reply per prompt
    max_workers=1   # Force Ragas to evaluate exactly 1 row at a time
)

results2 = evaluate(
    dataset=dataset3,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision,
        answer_correctness
        #ResponseGroundedness,
        #FactualCorrectness
    ],
    llm=local_judge_llm_llama3,
    embeddings=local_embeddings_ollama_nomic,
    run_config=local_run_config
)
df = results2.to_pandas()
display(df)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision,answer_correctness
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for automa...,Kubernetes is an open-source system for managi...,0.5,0.641003,1.0,1.0,0.744323
1,What is Docker?,[Kubernetes is an open-source container orches...,A Docker is a device that helps ship dock at t...,Docker is a platform for building and running ...,0.0,0.631352,1.0,0.5,0.714036
